In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import zscore
import matplotlib.dates as mdates
import os
import re

sns.set(style='white')  # light background
data_folder = './Data/clean/'
num = 80
file = f'd_{num}.csv'
path = os.path.join(data_folder, file)
Ticks = True  # Set to False if the file does not contain Ticks
# Load and clean data
# def clean_cell(cell):
#     # Keep only alphanumeric and spaces
#     return float(re.sub(r'[^a-zA-Z0-9]', '', str(cell)))
# def last_digits(cell, n=5):
#     match = re.search(r'(\d{%d})\D*$' % n, str(cell))
#     return match.group(1) if match else ''
#data = pd.read_csv(path, converters={'Ticks': lambda x: last_digits(x, 5)})
#data['Tick'] = (data['Tick'].apply(clean_cell))
data = pd.read_csv(path, delimiter = ';')
data.columns = data.columns.str.strip()

# Convert Tick to time
if Ticks:
    data['Tick'] = data['Tick'].astype(int)  # Ensure Tick is float
    data['Time'] = pd.to_timedelta(data['Tick'], unit='ms')
    data['TimePlot'] = pd.Timestamp("2023-01-01") + data['Time']
else:
    data['TimePlot'] = range(len(data))

# Check for missing values
print("\nMissing values per column:")
print(data.isnull().sum())

# Columns to visualize
#columns_to_plot = ['Throttle', 'Speed', 'Current', 'Voltage', 'Speed1']
columns_to_plot = ['Current', 'Voltage']
#data.dropna(subset=['TimePlot'] + columns_to_plot, inplace=True)

# Plot each column
for col in columns_to_plot:
    if col in data.columns:
        x = data['TimePlot']
        y = data[col]
        y_smooth = y.rolling(window=10).mean()

        # Stats
        avg = y.mean()
        med = y.median()

        # Outliers (Z-score method)
        z_scores = zscore(y)
        outliers = np.abs(z_scores) > 2

        # Start plotting
        fig, ax = plt.subplots(figsize=(12, 6))

        #Use numpy arrays for plotting with only first dimension (this is weird)
    

        # Smoothed line
        ax.plot(x, y_smooth, color='green', label=f'Smoothed (avg={avg:.2f}, med={med:.2f})')

        # Raw dashed line
        ax.plot(x,y, linestyle='--', color='red', alpha=0.5, label='Raw Values')

        # Outliers as blue dots
        ax.scatter(x[outliers], y[outliers], color='blue', label='Outliers')

        # Labeling and formatting
        ax.set_xlabel("Time (HH:MM:SS)", color='black')
        ax.set_ylabel(f"{col} vs Time", color='black')
        ax.set_title(f"{col} vs Time", color='black')
        ax.tick_params(axis='x', colors='black', rotation=45)
        ax.tick_params(axis='y', colors='black')

        ax.xaxis.set_major_locator(mdates.AutoDateLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))

        # Legend in black
        legend = ax.legend(loc='upper left')
        for text in legend.get_texts():
            text.set_color('black')

        plt.tight_layout()
        plt.show()
    else:
        print(f"Column '{col}' not found in dataset.")


ValueError: unit must not be specified if the input contains a str